## Work of Shishir Poudel (PySpark Version)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, mean, trim
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("ProductCleaning").getOrCreate()

df_product = spark.read.csv("../Coursework_dataset/products.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 17:44:23 WARN Utils: Your hostname, saag, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/04/14 17:44:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 17:44:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Viewing the table products table to look at the type of data present

In [2]:
df_product.show(5)

+----------+-------------------+-----------+-----------+----------+-----------+---------+---------+
|product_id|       product_name|   category|subcategory|unit_price|supplier_id|weight_kg|is_active|
+----------+-------------------+-----------+-----------+----------+-----------+---------+---------+
|prod_00001|  ProBook 15 Laptop|Electronics|    Laptops|  35976.37|   supp_016|     6.95|        1|
|prod_00002| UltraSlim Notebook|Electronics|    Laptops|  45460.16|   supp_085|     2.22|        1|
|prod_00003|      GamerX Laptop|       NULL|    Laptops|  13424.22|   supp_172|    14.61|        1|
|prod_00004|  BudgetBook Laptop|Electronics|    Laptops|  27883.23|   supp_186|    28.07|        1|
|prod_00005|ThinEdge Laptop Pro|Electronics|    Laptops|  29410.07|   supp_122|     7.37|        1|
+----------+-------------------+-----------+-----------+----------+-----------+---------+---------+
only showing top 5 rows


### View the total number of items in each columns

In [3]:
non_null_counts = df_product.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df_product.columns
])
non_null_df = non_null_counts.toPandas().T.reset_index()
non_null_df.columns = ['Column Name', 'Non-Null Count']
non_null_df

,Column Name,Non-Null Count
0,product_id,5000
1,product_name,5000
2,category,4980
3,subcategory,5000
4,unit_price,4950
5,supplier_id,5000
6,weight_kg,5000
7,is_active,5000


### Viewing which columns has null values

In [4]:
null_counts = df_product.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_product.columns
])
null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['Column Name', 'Null Count']
null_df

,Column Name,Null Count
0,product_id,0
1,product_name,0
2,category,20
3,subcategory,0
4,unit_price,50
5,supplier_id,0
6,weight_kg,0
7,is_active,0


### Viewing the categories  which are null

In [5]:
df_product.filter(col("category").isNull()).show(truncate=False)
# df_product.filter(col("category") == "Electronics").show()

+----------+---------------------------+--------+-----------+----------+-----------+---------+---------+
|product_id|product_name               |category|subcategory|unit_price|supplier_id|weight_kg|is_active|
+----------+---------------------------+--------+-----------+----------+-----------+---------+---------+
|prod_00003|GamerX Laptop              |NULL    |Laptops    |13424.22  |supp_172   |14.61    |1        |
|prod_00194|UltraView Phone Max        |NULL    |Phones     |16272.95  |supp_151   |29.7     |1        |
|prod_00264|UltraView Phone Value      |NULL    |Phones     |3723.16   |supp_018   |7.12     |1        |
|prod_00288|CameraPhone Ultra Core     |NULL    |Phones     |13045.68  |supp_091   |6.65     |1        |
|prod_00323|Phone Case Armor           |NULL    |Accessories|47552.52  |supp_117   |2.84     |1        |
|prod_00820|Blouse Formal White Max    |NULL    |Womens     |3424.06   |supp_043   |19.45    |1        |
|prod_00887|Saree Silk Elegant Elite   |NULL    |Womens

In [6]:
duplicated_reviews = df_product.join(
    df_product.groupBy("product_id").count().filter(col("count") > 1).select("product_id"),
    on="product_id", how="inner"
)
duplicated_reviews.sort("product_id").show()

+----------+------------+--------+-----------+----------+-----------+---------+---------+
|product_id|product_name|category|subcategory|unit_price|supplier_id|weight_kg|is_active|
+----------+------------+--------+-----------+----------+-----------+---------+---------+
+----------+------------+--------+-----------+----------+-----------+---------+---------+



### Since only 20 products does not have categories, we can set it manually

In [7]:
# Viewing all the different types of categories
df_product.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|     Toys & Games|
|Sports & Outdoors|
|      Electronics|
|         Clothing|
|    Home & Garden|
|   Food & Grocery|
|            Books|
|  Beauty & Health|
|             NULL|
+-----------------+



In [8]:
# Create a mapping from subcategory to category for the nulls
category_mapping = {
    'Laptops': 'Electronics',
    'Phones': 'Electronics',
    'Accessories': 'Electronics',
    'Womens': 'Clothing',
    'Furniture': 'Home & Garden',
    'Kitchenware': 'Home & Garden',
    'Garden': 'Home & Garden',
    'Bedding': 'Home & Garden',
    'Non-Fiction': 'Books',
    'Gym': 'Sports & Outdoors',
    'Cycling': 'Sports & Outdoors',
    'Running': 'Sports & Outdoors',
    'Supplements': 'Beauty & Health',
    'Wellness': 'Beauty & Health',
    'Drinks': 'Food & Grocery'
}

mapping_expr = F.create_map([F.lit(x) for x in sum(category_mapping.items(), ())])

# Fill null 'category' values based on the 'subcategory' using the mapping
df_product = df_product.withColumn(
    "category",
    when(col("category").isNull(), mapping_expr[col("subcategory")])
    .otherwise(col("category"))
)

# Verify that the nulls have been filled in the 'category' column
df_product.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_product.columns
]).show()

+----------+------------+--------+-----------+----------+-----------+---------+---------+
|product_id|product_name|category|subcategory|unit_price|supplier_id|weight_kg|is_active|
+----------+------------+--------+-----------+----------+-----------+---------+---------+
|         0|           0|       0|          0|        50|          0|        0|        0|
+----------+------------+--------+-----------+----------+-----------+---------+---------+



### Viewing all the rows where unit price is null

In [9]:
null_price_count = df_product.filter(col("unit_price").isNull()).count()
df_product.filter(col("unit_price").isNull()).show(n=null_price_count, truncate=False)

+----------+----------------------------+-----------------+------------+----------+-----------+---------+---------+
|product_id|product_name                |category         |subcategory |unit_price|supplier_id|weight_kg|is_active|
+----------+----------------------------+-----------------+------------+----------+-----------+---------+---------+
|prod_00147|PowerStation Laptop Neo     |Electronics      |Laptops     |NULL      |supp_020   |17.8     |1        |
|prod_00245|MiniCall Phone Prime        |Electronics      |Phones      |NULL      |supp_169   |14.74    |1        |
|prod_00257|DualSIM Pro SE              |Electronics      |Phones      |NULL      |supp_004   |21.7     |1        |
|prod_00453|Phone Car Mount One         |Electronics      |Accessories |NULL      |supp_119   |23.11    |1        |
|prod_00663|Hoodie Pullover SE          |Clothing         |Mens        |NULL      |supp_115   |15.89    |1        |
|prod_00857|Saree Silk Elegant Plus     |Clothing         |Womens      |

### We can remove the rows with NaN values but before that we should check it it's not being used in any other tables

In [10]:
df_orders_items = spark.read.csv("../Coursework_dataset/order_items.csv", header=True, inferSchema=True)
df_orders_items.show(5)

+-------------+----------+----------+--------+------------------+----------+
|order_item_id|  order_id|product_id|quantity|unit_price_at_sale|line_total|
+-------------+----------+----------+--------+------------------+----------+
| item_0000001|ord_000001|prod_01446|       4|          16951.04|  67804.16|
| item_0000002|ord_000002|prod_04562|       5|           1449.13|   7245.65|
| item_0000003|ord_000002|prod_04575|       1|            782.99|    782.99|
| item_0000004|ord_000003|prod_03437|       3|            219.07|    657.21|
| item_0000005|ord_000003|prod_03429|       1|           2231.91|   2231.91|
+-------------+----------+----------+--------+------------------+----------+
only showing top 5 rows


In [11]:
product_id_to_check = 'prod_00147'

df_orders_items.filter(col("product_id") == product_id_to_check).show()

+-------------+----------+----------+--------+------------------+----------+
|order_item_id|  order_id|product_id|quantity|unit_price_at_sale|line_total|
+-------------+----------+----------+--------+------------------+----------+
| item_0000486|ord_000221|prod_00147|       4|             480.0|    1920.0|
| item_0002304|ord_001057|prod_00147|       2|             425.0|     850.0|
| item_0003121|ord_001441|prod_00147|       3|             485.0|    1455.0|
| item_0004022|ord_001855|prod_00147|       4|             425.0|    1700.0|
| item_0004496|ord_002068|prod_00147|       3|             385.0|    1155.0|
| item_0007651|ord_003537|prod_00147|       3|             460.0|    1380.0|
| item_0008544|ord_003947|prod_00147|       4|             385.0|    1540.0|
| item_0012316|ord_005657|prod_00147|       3|             415.0|    1245.0|
| item_0013683|ord_006286|prod_00147|       1|             440.0|     440.0|
| item_0020244|ord_009284|prod_00147|       5|             405.0|    2025.0|

### Convinently, we find out that the product with no unit price was bought by others,so we cannot remove the rows with NaN values.

### The order_items table has unit_price_at_sale column. We can get mean of this column and put it in the products table

In [12]:
# Clean product_id columns by stripping whitespace
df_product = df_product.withColumn("product_id", trim(col("product_id")))
df_orders_items = df_orders_items.withColumn("product_id", trim(col("product_id")))

# Calculate the mean unit_price_at_sale for each product_id in df_orders_items
mean_prices = df_orders_items.groupBy("product_id") \
    .agg(mean("unit_price_at_sale").alias("mean_price"))

# Fill NaN values in df_product['unit_price'] using the calculated mean prices
df_product = df_product.join(mean_prices, on="product_id", how="left")
df_product = df_product.withColumn(
    "unit_price",
    when(col("unit_price").isNull(), col("mean_price"))
    .otherwise(col("unit_price"))
).drop("mean_price")

# Verify that the nulls have been filled in the 'unit_price' column.
df_product.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_product.columns
]).show()

+----------+------------+--------+-----------+----------+-----------+---------+---------+
|product_id|product_name|category|subcategory|unit_price|supplier_id|weight_kg|is_active|
+----------+------------+--------+-----------+----------+-----------+---------+---------+
|         0|           0|       0|          0|         1|          0|        0|        0|
+----------+------------+--------+-----------+----------+-----------+---------+---------+



### 1 product still has NaN unit_price after mapping, it means they were not present in df_orders_items with a unit_price_at_sale.

In [13]:
# For any remaining NaN unit_price (e.g., product not in order_items), fill with the overall mean of unit_price
overall_mean_unit_price = df_product.select(mean("unit_price")).collect()[0][0]
df_product = df_product.withColumn(
    "unit_price",
    when(col("unit_price").isNull(), F.lit(overall_mean_unit_price))
    .otherwise(col("unit_price"))
)

In [14]:
df_product.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_product.columns
]).show()

+----------+------------+--------+-----------+----------+-----------+---------+---------+
|product_id|product_name|category|subcategory|unit_price|supplier_id|weight_kg|is_active|
+----------+------------+--------+-----------+----------+-----------+---------+---------+
|         0|           0|       0|          0|         0|          0|        0|        0|
+----------+------------+--------+-----------+----------+-----------+---------+---------+



### Let's Check if there is any anomolies in the datatypes

In [15]:
df_product.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- is_active: integer (nullable = true)



### Hence, we conclude that the products table is clean

In [16]:
df_product.write.csv("products_cleaned.csv", header=True, mode="overwrite")
print("df_product exported successfully to 'products_cleaned.csv'")

df_product exported successfully to 'products_cleaned.csv'
